# Step 15.2 — Why Autoregressive GPT Can Be Decoder-Only：为什么语言模型不再需要 Separate Encoder？

Step 15.1 的 Original Transformer 有两条独立输入流：

```text
source sequence -> encoder memory
target sequence -> decoder
                     ↑
               cross-attention
```

而 GPT 面对的核心训练目标不是“给定一条独立 source sequence，再生成另一条 target sequence”，而是：

> **在同一条 token stream 上，根据已经出现的 prefix 预测下一个 token。**

这会让 architecture 发生一个非常重要的简化：

```text
separate encoder        removed
cross-attention         removed
causal self-attention   kept
one token stream        kept
```

本节只解释为什么这种 decoder-only 结构对 autoregressive language modeling 是充分的。

本轮不进入 Step 15.3，不列 MiniGPT 顶层模块，也不实现 GPT model。


## 1. 先从任务本身开始：GPT 要建模的是一个 Sequence Distribution

给定一条 token sequence：

$$
x_0,x_1,\ldots,x_T.
$$

Autoregressive factorization：

$$
\boxed{
p(x_0,x_1,\ldots,x_T)
=
\prod_{t=0}^{T}
p(x_t\mid x_{<t})
}.
$$

其中：

$$
x_{<t}
=
(x_0,\ldots,x_{t-1}).
$$

这意味着预测 token `x_t` 时，模型需要的信息集合就是：

> **同一条 stream 中的 earlier tokens。**

没有额外规定：

```text
必须先有一个独立 source encoder
必须生成 encoder memory
必须再通过 cross-attention 查询它
```

这些是 encoder–decoder conditional generation architecture 的结构，而不是 autoregressive factorization 本身的数学要求。


## 2. Training Input / Label 是 Shifted Views，不是 Source / Target 两条架构流

假设完整 sequence：

```text
<BOS>  I  love  coffee  <EOS>
```

next-token training 构造：

```text
input : <BOS>  I     love   coffee
label : I      love  coffee <EOS>
```

这里 input 与 label 的确看起来像“两排序列”，但它们只是：

> **同一个 underlying token stream 的两个 shifted views，用来定义 supervision。**

它们不是 Original Transformer 那种：

```text
source sentence -> encoder
target sentence -> decoder
```

两套独立 architecture inputs。

因此不要把 next-token label tensor 误认成“GPT 的 decoder target stream”，再凭空寻找一个 encoder source stream。


## 3. Causal Self-Attention 已经恰好提供 `p(next | prefix)` 所需的信息

训练 input positions：

$$
x_0,x_1,\ldots,x_{T-1}.
$$

位置 `i` 的 representation 用来预测：

$$
x_{i+1}.
$$

因此 position `i` 合法读取：

$$
x_0,\ldots,x_i.
$$

也就是 causal rule：

$$
\boxed{j\le i}.
$$

所以 causal self-attention 的 information set 正好是：

$$
\boxed{
\text{visible context at position }i
=
\text{prefix available before predicting }x_{i+1}
}.
$$

这就是 decoder-only 对 autoregressive LM 足够的最核心原因。

模型不需要 cross-attention 去读取另一份 memory，因为它需要的上下文本来就在自己的 prefix token stream 里。


## 4. Tiny Visibility Example：一条 Stream 自己就是 Context Memory

取训练 input：

```text
position:   0      1      2       3
token:     <BOS>   I     love    coffee
target:      I    love   coffee  <EOS>
```

Causal visibility：

```text
Query 0 -> Keys [0]
Query 1 -> Keys [0,1]
Query 2 -> Keys [0,1,2]
Query 3 -> Keys [0,1,2,3]
```

所以 Query 2 在预测 `coffee` 时已经可以读取：

```text
<BOS> I love
```

Query 3 在预测 `<EOS>` 时可以读取：

```text
<BOS> I love coffee
```

这里每个 prefix representation set 就承担了上下文来源。

没有必要再把同样的 earlier tokens 送进另一套 encoder，只为了让 decoder cross-attend 回来。


In [ ]:
import mlx.core as mx

T = 4

# True means this Query/Key pair is allowed by causal visibility.
query_index = mx.arange(T)[:, None]
key_index = mx.arange(T)[None, :]
allowed = key_index <= query_index

print('allowed visibility matrix:')
print(allowed)

# Expected logical pattern:
# [[True,  False, False, False],
#  [True,  True,  False, False],
#  [True,  True,  True,  False],
#  [True,  True,  True,  True ]]
#
# This master notebook remains unexecuted.


## 5. 从 Original Transformer 做“架构减法”

Step 15.1 的 decoder layer：

```text
masked self-attention
↓
cross-attention to encoder memory
↓
FFN
```

对于 pure autoregressive language modeling，没有独立 source sequence / encoder memory。

于是 cross-attention 没有外部 memory 可以读取。

架构可以简化为：

```text
causal self-attention
↓
FFN
```

再配上：

- residual connections；
- normalization；
- position information。

也就是说 GPT-style decoder-only block **不是把 2017 decoder layer 原封不动保留下来**。

它保留 decoder 的 causal self-attention lineage，但移除了 encoder–decoder cross-attention，因为 separate encoder 已经不存在。


## 6. 为什么还叫 Decoder-Only？

这个术语容易造成一个误解：

> “Decoder-only = 把 Original Transformer decoder 整层一模一样复制下来。”

并不准确。

更好的历史理解是：

```text
Original Transformer decoder side
  masked self-attention
  cross-attention
  FFN

             ↓ remove encoder-conditioned branch

GPT-style decoder-only stack
  causal self-attention
  FFN
```

它继承的是：

> **autoregressive / masked self-attention 的 decoder-side computation pattern。**

而不是保留原版 decoder 的每一个 sublayer。


## 7. Decoder-Only 并不意味着“不能被条件输入控制”

假设想做一个 source-conditioned task，也可以把 source / instruction 放到同一 token stream 的前缀。

例如：

```text
<BOS>
Translate to Chinese:
I love coffee.
<SEP>
我 爱 咖啡 。
<EOS>
```

在生成中文 continuation 时，前面的 instruction + English source 全都已经位于 causal past。

所以 response position 可以通过 causal self-attention 读取整个 prompt prefix。

结构变成：

```text
[ prompt / source tokens | response tokens ]
            one causal token stream
```

而不是：

```text
source -> separate encoder memory
response -> decoder cross-attention
```

这解释了为什么 decoder-only model 仍然可以做大量 prompt-conditioned generation。


## 8. 但这不意味着 Encoder–Decoder Architecture 没有价值

Decoder-only 能把很多 conditional tasks 序列化到一个 stream 中，并不意味着：

- encoder–decoder Transformer 已经数学上多余；
- decoder-only 在所有 Seq2Seq 任务上都一定更优；
- source / target 明确分离的 inductive bias 永远没有价值。

Encoder–decoder architecture 可以显式地：

- 对完整 source 做 non-causal encoding；
- 保存独立 source memory；
- 让 target decoder cross-attend 到 source representations。

Decoder-only 则选择：

> 把条件上下文与 continuation 统一表示成一个 autoregressive prefix stream。

这是 architecture / modeling choice，不是“旧架构错误、新架构唯一正确”。


## 9. Prompt + Continuation 在概率上仍是一条 Autoregressive Sequence

把 prompt token sequence 记为：

$$
c_0,\ldots,c_{m-1}
$$

continuation：

$$
y_0,\ldots,y_{n-1}.
$$

把它们串起来：

$$
s
=
[c_0,\ldots,c_{m-1},y_0,\ldots,y_{n-1}].
$$

Autoregressive model 仍然建模：

$$
p(s)
=
\prod_t p(s_t\mid s_{<t}).
$$

当生成 `y_k` 时：

$$
s_{<t}
$$

已经包含：

- 完整 prompt prefix；
- 之前已经生成的 `y_{<k}`。

所以条件生成可以写成：

$$
p(y_k
\mid
c_0,\ldots,c_{m-1},
y_0,\ldots,y_{k-1}).
$$

这正是 causal self-attention 能表示的信息依赖。


## 10. Encoder–Decoder 与 Decoder-Only 的 Shape 对照

### Encoder–Decoder

```text
source
(B,T_src,C)
↓ encoder
memory
(B,T_src,C)

target
(B,T_tgt,C)
↓ masked self-attn
↓ cross-attn with memory
(B,T_tgt,C)
```

Cross-attention scores：

$$
(B,H,T_{tgt},T_{src}).
$$

### Decoder-Only

```text
one stream
(B,T,C)
↓ causal self-attention
(B,T,C)
```

Self-attention scores：

$$
(B,H,T,T)
$$

配 causal upper-triangle mask。

这里的 T 可以包含 prompt tokens + continuation-so-far tokens。

因此模型只需要维护一个 evolving sequence representation。


In [ ]:
# ARCHITECTURE-LEVEL SHAPE COMPARISON ONLY.

B = 2
H = 4
T_src = 5
T_tgt = 3
T_stream = 8
C = 16

encoder_decoder_cross_scores = (B, H, T_tgt, T_src)
decoder_only_self_scores = (B, H, T_stream, T_stream)

print('encoder-decoder cross-attention scores:', encoder_decoder_cross_scores)
print('decoder-only causal self-attention scores:', decoder_only_self_scores)

# These are expected architecture shapes, not runtime validation.
# No GPT model is implemented in this notebook.


## 11. 从 2017 Transformer 到 GPT：现在只做概念上的 Keep / Remove

### 保留（Keep）

- token representations；
- positional information 的需要；
- Multi-Head Self-Attention；
- causal mask；
- Feed-Forward MLP；
- residual connections；
- normalization；
- stacked Transformer blocks 的基本思想。

### 移除（Remove）

- separate source encoder stack；
- encoder memory；
- encoder–decoder cross-attention。

### 改成（Reframe）

```text
source + target
two architectural streams

          ↓

one autoregressive token stream
prefix -> next-token prediction
```

这里只是 architecture reasoning。

`token embedding / position embedding / Block × N / final norm / LM head` 的具体顶层 MiniGPT 结构属于 Step 15.3，本节不展开。


## 12. Training Parallelism 与 Generation Seriality 仍然同时存在

训练时：

```text
input : x0 x1 x2 ... x_(T-1)
label : x1 x2 x3 ... x_T
```

所有 input positions 可以一起送入 causal Transformer。

Mask 保证 position i 不能读取未来 positions。

因此：

$$
\text{parallel position computation}
\neq
\text{future visibility}.
$$

而 inference 时，真正未来 token 还不存在，所以：

```text
prefix
↓
predict next token
↓
append token
↓
predict next token
```

仍然是 autoregressive serial generation。

Decoder-only 并没有消除 task-level causal dependency。


## 13. Step 15.2 材料边界

- GPT 的核心任务是同一 token stream 上的 autoregressive next-token modeling：`p(x_0,...,x_T)=∏ p(x_t|x_<t)`。
- next-token input / labels 是同一 underlying sequence 的 shifted supervision views，不是 Original Transformer 的 separate source / target architecture streams。
- causal self-attention 让 position i 读取 `x_0...x_i` 来预测 `x_(i+1)`，因此 prefix context 已经存在于同一 stream 中。
- 没有 independent source encoder memory 时，encoder stack 与 encoder–decoder cross-attention 可以从 autoregressive LM architecture 中移除。
- decoder-only 不代表复制原始 decoder layer；GPT-style block 保留 causal self-attention + FFN lineage，但删除 cross-attention。
- prompt / source 可以序列化在 response 之前，使 generated tokens 通过 causal self-attention 读取完整 prefix，因此 decoder-only 仍可进行 conditional generation。
- 这不证明 decoder-only 在所有 Seq2Seq 任务上都优于 encoder–decoder architecture；二者是不同 modeling choices。
- training 可并行计算各 positions，但 mask 限制 future information；autoregressive inference 仍然逐 token。
- 本轮只准备 Step 15.2；不进入 Step 15.3，不列出或实现 MiniGPT 顶层的 token embedding、position embedding、Block × N、final norm、LM head。
